# Face Mask Detection using Transfer Learning (VGG16)

A binary image classification project that detects whether a person is wearing a face mask or not.  
Uses **VGG16** pretrained on ImageNet as a feature extractor, with a custom classification head trained on ~1,376 images.

**Key Concepts Practiced:**
- Transfer Learning (freezing base, training head)
- Data Augmentation (rotation, flip, zoom, shear)
- EarlyStopping & ModelCheckpoint callbacks
- Train / Validation / Test split (70/15/15)

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 2. Load & Prepare the Dataset

The dataset has two folders: `with_mask` and `without_mask`.  
Each image is resized to **224x224** (VGG16's expected input size).

In [ ]:
# Clone the dataset (run once — Colab only)
# !git clone https://github.com/ricklon/pyimagesearch-face-mask-detector.git

dataset_path = "/content/pyimagesearch-face-mask-detector/dataset"
categories = ['without_mask', 'with_mask']   # 0 = no mask, 1 = mask

print("Classes:", os.listdir(dataset_path))

In [ ]:
data = []
for category in categories:
    path = os.path.join(dataset_path, category)
    label = categories.index(category)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))
        data.append([img, label])

print(f"Total images loaded: {len(data)}")

## 3. Shuffle, Split & Preprocess

- Shuffle to remove ordering bias
- Split into **70% train / 15% validation / 15% test**
- Apply VGG16's `preprocess_input` (subtracts ImageNet channel means)

In [ ]:
random.shuffle(data)

X = np.array([item[0] for item in data])
y = np.array([item[1] for item in data])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: no_mask={sum(y==0)}, mask={sum(y==1)}")

In [ ]:
# VGG16-specific preprocessing (subtracts ImageNet mean, BGR format)
X = preprocess_input(X)

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 4. Build the Model (VGG16 Transfer Learning)

**Strategy:**
1. Load VGG16 pretrained on ImageNet
2. Remove the final classification layer (1000-class softmax)
3. Freeze all VGG16 layers (use as feature extractor only)
4. Add a single `Dense(1, sigmoid)` layer for binary classification

This gives us ~134M frozen params + just 4,097 trainable params.

In [ ]:
vgg = VGG16()

# Build a Sequential model with all VGG16 layers except the last one
model = Sequential()
for layer in vgg.layers[:-1]:
    model.add(layer)

# Freeze all pretrained layers
for layer in model.layers:
    layer.trainable = False

# Add binary classification head
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train with Data Augmentation

- **ImageDataGenerator** applies random transforms on-the-fly (rotation, flip, zoom, shear)
- **EarlyStopping** stops training if validation loss doesn't improve for 3 epochs
- **ModelCheckpoint** saves the best model weights automatically

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

## 6. Evaluate on Test Set

Check classification report (precision, recall, F1) and plot training curves.

In [ ]:
# Classification report on the held-out test set
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred,
      target_names=['without_mask', 'with_mask']))

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. What I Learned

- **Transfer learning** is extremely powerful for small datasets — VGG16's pretrained features generalize well to face mask detection even though ImageNet doesn't have mask images.
- **Freezing layers** is key — training only the final dense layer keeps training fast and avoids overfitting on a small dataset.
- **Data augmentation** helps regularize by creating variations of existing images.
- **EarlyStopping** prevents wasting compute and overfitting by monitoring validation loss.
- For a dataset of ~1,376 images, a simple binary head on top of VGG16 achieves **~96% accuracy**.